# Retrace — Colab runner

Runs the GPU stages. Set the runtime to **T4 GPU** (Runtime → Change runtime type).

Every cell just calls `python -m retrace <stage>`; all logic lives in the `retrace/` package.

## 1. Code + install

In [ ]:
import os

REPO_URL = "https://github.com/Varshamani13/hackathon26.git"

if not os.path.isdir("hackathon26"):
    !git clone $REPO_URL
%cd hackathon26

!pip -q install -e ".[train,serve,report]"
# Colab pins we need: old torchao breaks transformers import; transformers 5.x
# changed the Trainer signature. Force a compatible 4.x line.
!pip -q uninstall -y torchao 2>/dev/null || true
!pip -q install "transformers>=4.44,<5" "peft>=0.12,<0.15"

!python -c "import torch, transformers, peft; print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| cuda', torch.cuda.is_available())"

## 2. Knowledge prep  (CPU, seconds)

In [ ]:
!python -m retrace -v prepare

## 3. Baseline training  (GPU, ~30 min)

LoRA fine-tune → merge → gate. The gate blocks only on **fact recall** (direct +
cloze ≥ 0.90, direct+cloze+boolean ≥ 0.75). Multi-hop / reverse-lookup are
reported but don't block — a 0.5B model trained on atomic facts isn't expected
to chain them.

Exit `0` = passed, `3` = failed → bump `--epochs` / `--lora-r`, or
`--base meta-llama/Llama-3.2-1B-Instruct`.
Already trained once? Re-run just the gate with `--skip-train` (no GPU retrain).

In [ ]:
!python -m retrace -v train --epochs 4 --lora-r 32

In [ ]:
import json
g = json.load(open("artifacts/baseline/gate_report.json"))
print("gate passed:", g["passed"])
for k, v in g["checks"].items():
    tag = "" if v.get("blocking", True) else "  (info only)"
    print(f"  {k:12s} {v['value']:.3f}  (>= {v['threshold']:.2f}){tag}")
print("\nby probe type:")
for t, s in g["overall"]["by_type"].items():
    print(f"  {t:16s} {s['accuracy']:.3f}  (n={s['n']})")

## 4. Erase → verify → report  (GPU, ~5 min)

Demo target: **NeuroSync Diagnostics** (G001). Its look-alikes NeuroWave / NeuroCore and the other Denver companies must survive.

In [ ]:
!python -m retrace -v pipeline "NeuroSync Diagnostics"

In [ ]:
import json
v = json.load(open("artifacts/verification/G001/verification.json"))
s = v["scores"]
print("Retrace score (weighted):      ", round(s["retrace_score_weighted"], 3))
print("forget efficacy:               ", round(s["forget_efficacy"], 3))
print("retain preservation:           ", round(s["retain_preservation"], 3))
print("adversarial resistance:        ", round(s["adversarial_resistance"], 3))
print("\nbehavioral accuracy (baseline -> erased):")
for k in ("forget", "retain_hard", "retain_broad", "capability"):
    b = v["behavioral"]["baseline"][k]["accuracy"]
    e = v["behavioral"]["erased"][k]["accuracy"]
    print(f"  {k:14s} {b:.2f} -> {e:.2f}")
print("\nlook-alikes:")
for n in v["neighborhood"]:
    print(f"  {n['entity']:26s} {n['baseline_acc']:.2f} -> {n['erased_acc']:.2f}")

In [ ]:
from IPython.display import Markdown
Markdown(open("artifacts/reports/G001/erasure_report.md").read())

## 5. Live demo (Streamlit + tunnel)

In [ ]:
!pip -q install pyngrok
from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_TOKEN")   # from dashboard.ngrok.com
public = ngrok.connect(8501)
print("demo:", public)
!streamlit run retrace/serving/app.py --server.port 8501 --server.headless true &> streamlit.log &

## 6. Back up to Drive (survive disconnects)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/retrace" && cp -r artifacts "/content/drive/MyDrive/retrace/"
print("backed up")